In [11]:
#!pip install google_cloud_pipeline_components

In [12]:
import components as components
import os
# Import necessary libraries
from kfp.v2 import dsl
from kfp.v2.dsl import component
from kfp.v2 import compiler
from google.cloud import aiplatform
from pipeline_helper import process_sql_file
import pickle as pk

In [13]:
constants = {
    "DATASET": "anbc-hcb-dev",
    "SCHEMA": "clin_analytics_hcb_dev",
    "prefix": "cp_strategic_stratification_test",
    "database_name": "anbc-hcb-dev.clin_analytics_hcb_dev",
    "dec_database_name": "anbc-hcb-dev.clin_analytics_hcb_dev",
    "SHARE_SCHEMA": "anbc-hcb-dev.clin_analytics_share_hcb_dev",
    "OWNER": "parked_aetna_com",
    "COSTCENTER": "13070",
    "EMAIL": "parked_aetna_com", # e.g., john.doe@cvshealth.com
    # "TENANT": "hcm-cm-de", # e.g., mleng-platform
    "USE_COMPUTE_PROJECT": True, # e.g., True or False
    "TENANT":"hcm-cm-de",
    "COMPUTE_PROJECT": "anbc-dev-hcm-cm-de",
    "PROJECT": "anbc-dev-hcm-cm-de",
}

service_account = "gchcb-hcm-cm-de-ontpd@anbc-dev-hcm-cm-de.iam.gserviceaccount.com"
decrypt_sa = "gchcb-hcm-cm-de-dec-ontpd@anbc-dev-hcm-cm-de.iam.gserviceaccount.com"
project=constants["PROJECT"]

In [14]:
from kfp import dsl
from kfp.dsl import (
    Input, Output, Dataset, Model, component, pipeline
)
from google.cloud import aiplatform
from typing import Any, Dict, List, Optional, Text, Tuple, Union, Sequence, Callable
import yaml
from kfp.v2 import compiler
@component(
    base_image="python:3.9",
    packages_to_install=[
        "google-cloud-aiplatform>=1.38.0",
        "google-cloud-bigquery>=2.0.0",
        "google-auth>=2.0.0",
        "requests>=2.25.0"
    ]
)
def vertex_batch_predict_bigquery_component(
    project: str,
    location: str,
    service_account: str,
    cmek_key: str,
    cost_center: str,
    owner: str,
    #Model details
    model_resource_name: str,
    # BigQuery specific
    key_field: str,  # Unique key field in input table
    input_table: str,  # project.dataset.table
    output_table: str, # project.dataset.table (final output)
    compute_dataset: str, #hcm_cm_de_dec_beam_{ENV}_hcm_cm_de"
    expiration_days: int = 30, # BQ table expiration in days
    # Instance configuration - field filtering
    excluded_fields: list = None,  # Fields to exclude from predictions
    included_fields: list = None,  # Fields to include (if specified, only these will be sent)
    # Selected features configuration
    selected_features: list = None,  # List of feature names in proper order for model input
    # Machine configuration
    machine_type: dict = {"machine_type": "n2-standard-64"},
    # Job configuration
    starting_replica_count: int = 1,
    max_replica_count: int = 1,
    batch_size: int = None,  # Auto-determined if None
) -> str:
    """
    Runs a Vertex AI batch prediction job with comprehensive parameter validation.
    
    Copies input table from shared project to compute project temp dataset,
    runs batch prediction to temp output table in compute project temp dataset, 
    then copies to final output table in shared project.

    Args:
        project: GCP project ID.
        location: GCP region.
        compute_dataset: temp dataset in compute project
        model_resource_name: Full resource name of the registered model.
        job_display_name: Display name for the batch prediction job.
        input_table: Input data table (BigQuery table).
        output_table: Output data table (BigQuery table).
        excluded_fields: List of field names to exclude from predictions.
        included_fields: List of field names to include (mutually exclusive with excluded_fields).
        selected_features: List of feature names in proper order. If provided, only these features
                          will be selected from input_table in the specified order. The key_field
                          will be included automatically if not in selected_features.
        machine_type: Machine type for batch prediction.
        expiration_days: Days until output table expires (default: 30).

    Returns:
        The resource name of the batch prediction job.
    """
    from google.cloud import aiplatform
    from google.cloud import bigquery
    import requests
    import json
    from google.auth import default
    from google.auth.transport.requests import Request
    # Initialize clients
    bq_client = bigquery.Client(project=project)
    
    # Setup table names and labels
    input_table_name = input_table.split(".")[-1]
    output_table_name = output_table.split(".")[-1]
    labels = f"""labels=[("owner","{owner}"),("costcenter","{cost_center}")]"""

    # Copy input table to temp dataset with selected features in proper order
    temp_input_table = f"{project}.{compute_dataset}.{input_table_name}_tmp"
    
    # Build SELECT clause based on selected_features
    if selected_features:
        # Ensure key_field is included if not already in selected_features
        columns_to_select = selected_features.copy()
        # if key_field not in columns_to_select:
        #     # Add key_field at the beginning to maintain it for joining results
        #     columns_to_select.insert(0, key_field)
        
        # Escape column names with backticks to handle special characters
        escaped_columns = [f"`{col}`" for col in columns_to_select]
        select_clause = ", ".join(escaped_columns)
        query = f"CREATE OR REPLACE TABLE {temp_input_table} OPTIONS({labels}, expiration_timestamp=TIMESTAMP_ADD(CURRENT_TIMESTAMP(), INTERVAL 5 DAY)) AS SELECT {select_clause} FROM `{input_table}` LIMIT 100"
    else:
        # Fallback to SELECT * if no selected_features provided (backward compatibility)
        query = f"CREATE OR REPLACE TABLE {temp_input_table} OPTIONS({labels}, expiration_timestamp=TIMESTAMP_ADD(CURRENT_TIMESTAMP(), INTERVAL 5 DAY)) AS SELECT * FROM `{input_table}` LIMIT 100"
    
    job = bq_client.query(query)
    job.result()

    # Prepare temp output table
    temp_output_table = f"{project}.{compute_dataset}.{output_table_name}_tmp"

    # Create batch prediction job using REST API
    
    # Get authentication credentials
    credentials, _ = default()
    credentials.refresh(Request())
    access_token = credentials.token
    
    # Extract project, location, and model ID from model resource name
    # Format: projects/{project}/locations/{location}/models/{model_id}
    model_parts = model_resource_name.split('/')
    model_project = model_parts[1]
    model_location = model_parts[3]
    model_id = model_parts[5]
    
    # Convert machine_type keys to camelCase for REST API
    def to_camel_case(snake_str):
        """Convert snake_case to camelCase"""
        components = snake_str.split('_')
        return components[0] + ''.join(word.capitalize() for word in components[1:])

    machine_type = {k: to_camel_case(v) for k, v in machine_type.items() if v is not None}
    
    # Build instance config if field filtering is specified
    instance_config = {}
    if excluded_fields:
        instance_config["excludedFields"] = excluded_fields
    elif included_fields:
        instance_config["includedFields"] = included_fields
    job_display_name = f"{model_id}_model_prediction"
    # Build the REST API request payload
    batch_prediction_request = {
        "displayName": job_display_name,
        "model": model_resource_name,
        "serviceAccount": service_account,
        "inputConfig": {
            "instancesFormat": "bigquery",
            "bigquerySource": {"inputUri": f"bq://{temp_input_table}"}
        },
        "outputConfig": {
            "predictionsFormat": "bigquery",
            "bigqueryDestination": {"outputUri": f"bq://{temp_output_table}"}
        },
        "dedicatedResources": {
            "machineSpec": machine_type,
            "startingReplicaCount": starting_replica_count,
            "maxReplicaCount": max_replica_count
        },
        "encryptionSpec": {
            "kmsKeyName": cmek_key
        },
        'instanceConfig': {
            'includedFields': selected_features,
        },
    }
    
    # Add optional parameters
    if batch_size:
        batch_prediction_request["manualBatchTuningParameters"] = {
            "batchSize": batch_size
        }
    if instance_config:
        batch_prediction_request["instanceConfig"] = instance_config
 
    print(json.dumps(batch_prediction_request, indent=2))
    # Make the REST API call (using v1beta1 to match documentation)
    # https://cloud.google.com/vertex-ai/docs/reference/rpc/google.cloud.aiplatform.v1beta1#batchpredictionjob
    # https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.BatchPredictionJob
    url = f"https://{model_location}-aiplatform.googleapis.com/v1beta1/projects/{model_project}/locations/{model_location}/batchPredictionJobs"
    
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }
    
    response = requests.post(url, headers=headers, json=batch_prediction_request)
    
    if response.status_code != 200:
        raise Exception(f"Batch prediction job creation failed: {response.status_code} - {response.text}")
    
    batch_job_response = response.json()
    batch_predict_job_name = batch_job_response["name"]
    
    print(f"Batch prediction job created: {batch_predict_job_name}")
    
    # use Python SDK for job waiting
    aiplatform.init(project=project, location=location, service_account=service_account)
    batch_job = aiplatform.BatchPredictionJob(batch_predict_job_name)
    batch_job.wait_for_completion()  # Built-in exponential backoff and error handling
    
    print(f"Batch prediction job completed with state: {batch_job.state}")

    # Copy prediction table to final output table
    if expiration_days:
        options=f'{labels}, expiration_timestamp=TIMESTAMP_ADD(CURRENT_TIMESTAMP(), INTERVAL {expiration_days} DAY)'
    else:
        options=labels
    query = f"CREATE OR REPLACE TABLE {output_table} OPTIONS({options}) AS SELECT * FROM `{temp_output_table}`"
    copy_pred_job = bq_client.query(query)
    copy_pred_job.result()
    #clean up temp tables
    bq_client.delete_table(temp_input_table, not_found_ok=True)
    bq_client.delete_table(temp_output_table, not_found_ok=True)

    return batch_predict_job_name


In [15]:
# In pipeline.ipynb - Update the batch_prediction_pipeline function
@dsl.pipeline(
    name="batch-prediction-pipeline",
    description="Pipeline for Vertex AI batch prediction with BigQuery input/output"
)
def batch_prediction_pipeline(
    project: str,
    location: str,
    service_account: str,
    cmek_key: str,
    cost_center: str,
    owner: str,
    # Model details
    model_resource_name: str,
    # BigQuery specific
    key_field: str,
    input_table: str,
    output_table: str,
    compute_dataset: str,
    expiration_days: int = 30,
    # Instance configuration - field filtering
    excluded_fields: list = None,
    included_fields: list = None,
    # Selected features configuration
    selected_features: list = None,  # List of feature names in proper order
    # Machine configuration
    machine_type: dict = None,
    # Job configuration
    starting_replica_count: int = 1,
    max_replica_count: int = 1,
    batch_size: int = None,
):
    # Batch prediction component
    train_selected_features = ['x','y','z']
    batch_pred_job = vertex_batch_predict_bigquery_component(
        project=project,
        service_account=service_account,
        cmek_key=cmek_key,
        location=location,
        cost_center=cost_center,
        owner=owner,
        model_resource_name=model_resource_name,
        key_field=key_field,
        input_table=input_table,
        output_table=output_table,
        compute_dataset=compute_dataset,
        expiration_days=expiration_days,
        excluded_fields=excluded_fields,
        included_fields=included_fields,
        selected_features=train_selected_features,  # Add this line
        machine_type=machine_type or {"machine_type": "n1-standard-16"},
        starting_replica_count=starting_replica_count,
        max_replica_count=max_replica_count,
        batch_size=batch_size,
    )

In [16]:
# Define the pipeline
@dsl.pipeline(
    name="strategic-stratification-cp-bigquery-inference-pipeline",
    description="Pipeline to perform BigQuery operations in sequential order"
)
def bigquery_pipeline(
    project_id: str = "anbc-dev-hcm-cm-de",
    region: str = "us-east4",
    cmek_key: str = "projects/cvs-key-vault-nonprod/locations/us-east4/keyRings/gkr-nonprod-us-east4/cryptoKeys/gk-anbc-dev-hcm-cm-de-us-east4"
):

    
    
    # combine runs
    
    # generate post data for testing
    
    # create_cohort = process_sql_file('cohort.sql', constants, base_path="../sql_files_prod/") # cohort
    # run_create_cohort = components.bq_query_with_impersonation(query=create_cohort, project=project)
    # run_create_cohort.set_display_name("create_cohort")

    # claims = process_sql_file('claims.sql', constants, base_path="../sql_files_prod/") # claims
    # run_claims = components.bq_query_with_impersonation(query=claims, project=project)
    # run_claims.set_display_name("claims")

    # medical_case = process_sql_file('med_case.sql', constants, base_path="../sql_files_prod/") # med case
    # run_medical_case = components.bq_query_with_impersonation(query=medical_case, project=project)
    # run_medical_case.set_display_name("medical_case")
    
    # HPD = process_sql_file('HPD.sql', constants, base_path="../sql_files_prod/") # HPD
    # run_HPD = components.bq_query_with_impersonation(query=HPD, project=project)
    # run_HPD.set_display_name("HPD")

    # physician_visits = process_sql_file('physician_visit_stats.sql', constants, base_path="../sql_files_prod/") # phys vis stats
    # run_physician_visits = components.bq_query_with_impersonation(query=physician_visits, project=project)
    # run_physician_visits.set_display_name("physician_visits")

    # cost_cv = process_sql_file('cost_cv.sql', constants, base_path="../sql_files_prod/") # cost cv
    # run_cost_cv = components.bq_query_with_impersonation(query=cost_cv, project=project)
    # run_cost_cv.set_display_name("cost_cv")

    # analytic = process_sql_file('analytic.sql', constants, base_path="../sql_files_prod/")    # analytic
    # run_analytic = components.bq_query_with_impersonation(query=analytic, project=project)
    # run_analytic.set_display_name("analytic")

    # combine = process_sql_file('combine_model_runs.sql', constants, base_path="../sql_files_prod/")    # model run combine
    # run_combine = components.bq_query_with_impersonation(query=analytic, project=project)
    # run_combine.set_display_name("combine")
    
    cancer_inference = vertex_batch_predict_bigquery_component(
        project=project_id,
        service_account=service_account,
        cmek_key=cmek_key,
        location=region,
        cost_center= constants["COSTCENTER"],
        owner=constants["OWNER"],
        model_resource_name="projects/46378383599/locations/us-east4/models/7593434009607077888",
        key_field="individual_id", 
        input_table="anbc-hcb-dev.clin_analytics_hcb_dev.cp_strategic_stratification_test_analytic",  # project.dataset.table
        output_table="anbc-hcb-dev.clin_analytics_hcb_dev.cp_strategic_stratification_test_cancer_model", # project.dataset.table (final output)
        compute_dataset="hcm_cm_de_beam_dev_hcm_cm_de", #hcm_cm_de_dec_beam_{ENV}_hcm_cm_de"
        ## add function for how to reorder table
        expiration_days=30, # output table expiration in days
        ## done in this way so we don't have to port the files around, would obviously rather do it that way
        selected_features = ['emb131', 'emb169', 'emb186', 'uniq_dx_cd_cnt_3mo', 'emb14', 'change_q1_minus_q_2', 'Oncologic_case_pre_12', 
                             'other_spec_visits_pre_1', 'cost_9_12', 'pcp_visits_pre_9', 'spec_visits_pre_6', 'emb194', 'emb116', 'lab_max_psa_1yr', 
                             'emb94', 'emb238', 'emb125', 'emb190', 'dx_cd_cnt_6mo', 'Endocrine_Metabolic_case_pre_12', 'emb42', 'emb166', 'emb89', 
                             'cancer_cv', 'Cardiac_case_pre_12', 'spec_visit_cnt_6mo', 'emb248', 'cancer_visits_pre_9', 'prostate_cancer', 'emb145', 
                             'emb152', 'Respiratory_case_pre_12', 'emb250', 'cost_6_9', 'emb176', 'emb222', 'endo_cv', 'cancer_visits_pre_12', 'emb61', 
                             'emb6', 'emb80', 'emb74', 'emb118', 'emb233', 'emb46', 'spec_visits_pre_1', 'emb92', 'emb57', 'emb66', 'emb197', 'emb244', 
                             'pcp_visits_pre_1', 'emb91', 'colorectal_cancer', 'lab_max_psa_2yr', 'emb170', 'lab_visit_cnt_2yr', 'Musculoskeletal_case_pre_12', 
                             'spec_visits_pre_12', 'emb185', 'emb31', 'age', 'emb225', 'emb202', 'change_q1_minus_q_3', 'Digestive_case_pre_12', 'emb86', 
                             'change_h_1_minus_h_2', 'emb149', 'lab_visit_cnt_3mo', 'spec_visits_pre_9', 'emb29', 'card_visits_pre_12', 'leukemia_myeloma', 
                             'emb110', 'health_habits', 'emb35', 'breast_cancer', 'emb219', 'cancer_visits_pre_3', 'emb242', 'emb224', 'cancer', 'dx_diff', 
                             'Oncologic_case_pre_3', 'emb21', 'cost_0_3', 'pcp_cv', 'emb187', 'emb73', 'emb175', 'social_risk_score', 'emb142', 'emb2', 'brain_cancer', 
                             'uniq_proc_cd_cnt_6mo', 'emb62', 'clm_ln_cnt_2yr', 'emb160', 'Obstetric_case_pre_12', 'emb24', 'emb37', 'emb68', 'language_score', 
                             'uniq_rev_cd_cnt_3mo', 'emb77', 'cost_3_6', 'emb128', 'emb25', 'cancer_visits_pre_1', 'clm_ln_cnt_1yr', 'emb214', 'er_clm_cnt_6mo', 
                             'lab_visit_cnt_6mo', 'emb41', 'emb75', 'emb137', 'emb234', 'emb85', 'emb56', 'emb207', 'Injury_Poisoning_case_pre_12', 'emb203', 
                             'rx_diff', 'emb32', 'emb182', 'emb232', 'uniq_proc_cd_cnt_3mo', 'emb228', 'Renal_case_pre_12', 'Neurologic_case_pre_12', 'emb33', 
                             'index_month', 'emb71', 'hpd_count', 'emb143', 'emb235', 'emb159', 'emb247', 'emb120', 'emb236', 'other_cancer', 'Infectious_case_pre_12', 
                             'emb45', 'emb64', 'emb181', 'emb180', 'emb124', 'pcp_visits_pre_6', 'Oncologic_case_pre_6', 'emb114', 'emb220', 'emb227', 'ortho_visits_pre_3', 
                             'change_q1_minus_q_4', 'emb147'],
        machine_type={"machine_type": "n2-standard-64"},
        starting_replica_count=1,
        max_replica_count=10,
    )

    # cardiac_inference = vertex_batch_predict_bigquery_component(
    #     project=project_id,
    #     service_account=service_account,
    #     cmek_key=cmek_key,
    #     location=region,
    #     cost_center= constants["COSTCENTER"],
    #     owner=constants["OWNER"],
    #     model_resource_name="projects/46378383599/locations/us-east4/models/8244204155762114560",
    #     key_field="individual_id", 
    #     input_table="anbc-hcb-dev.clin_analytics_hcb_dev.cp_strategic_stratification_test_analytic",  # project.dataset.table
    #     output_table="anbc-hcb-dev.clin_analytics_hcb_dev.cp_strategic_stratification_test_cardiac_model", # project.dataset.table (final output)
    #     compute_dataset="hcm_cm_de_beam_dev_hcm_cm_de", #hcm_cm_de_dec_beam_{ENV}_hcm_cm_de"
    #     ## add function for how to reorder table
    #     expiration_days=30, # output table expiration in days
    #     excluded_fields=["individual_id"],  # Fields to include
    #     selected_features = ['emb177', 'emb54', 'pcp_cv', 'emb41', 'gender_cd', 'emb164', 'Obstetric_case_pre_12', 'emb18', 'pharmacy_days_supply_1yr_cnt', 
    #                          'emb179', 'uniq_rev_cd_cnt_2yr', 'mem_mos_12_pre', 'cost_0_3', 'uniq_proc_cd_cnt_6mo', 'emb31', 'uniq_dx_cd_cnt_6mo', 'emb210', 
    #                          'emb151', 'Respiratory_case_pre_12', 'Neurologic_case_pre_6', 'congential_heart_disease', 'skin_cancer', 'emb130', 'card_cv', 
    #                          'emb50', 'ortho_cv', 'emb87', 'emb234', 'dx_cd_cnt_6mo', 'pharmacy_days_supply_6mo_cnt', 'lab_max_glucose_2yr', 'ischemic_heart_disease', 
    #                          'bh', 'cerebrovascular_condition', 'card_visits_pre_3', 'emb105', 'emb229', 'hpd_major_flag', 'pharmacy_gpi4_3mo_count', 'language_score', 
    #                          'emb154', 'change_q1_minus_q_4', 'emb142', 'emb99', 'emb218', 'emb36', 'drug_ind', 'heart_failure', 'emb195', 'index_month', 
    #                          'atrial_fibrillation', 'emb121', 'emb100', 'health_acess', 'emb225', 'lab_max_cholest_2yr', 'card_visits_pre_9', 
    #                          'dx_diff', 'spec_cv', 'uniq_dx_cd_cnt_1yr', 'emb94', 'pharmacy_gpi4_6mo_count', 'ventricular_arrhythmia', 'emb79', 
    #                          'emb156', 'mm_2yr_cnt', 'emb212', 'emb112', 'emb48', 'emb111', 'uniq_dx_cd_cnt_3mo', 'emb88', 'age', 'emb134', 
    #                          'Neurologic_case_pre_12', 'emb131', 'er_clm_cnt_1yr', 'emb90', 'emb21', 'emb47', 'Oncologic_case_pre_12', 'Cardiac_case_pre_3', 
    #                          'emb44', 'emb140', 'emb205', 'emb106', 'Injury_Poisoning_case_pre_12', 'emb232', 'emb240', 'emb128', 'emb124', 'emb113', 
    #                          'Musculoskeletal_case_pre_12', 'emb76', 'emb181', 'change_q1_minus_q_3', 'Renal_case_pre_12', 'emb182', 'card_visits_pre_6', 
    #                          'inflammatory_bowel_disease', 'er_clm_cnt_6mo', 'emb96', 'Digestive_case_pre_12', 'emb126', 'Infectious_case_pre_12', 
    #                          'uniq_rev_cd_cnt_1yr', 'cancer', 'lab_max_creat_1yr', 'emb52', 'emb224', 'emb245', 'card_visits_pre_1', 'lab_max_creat_2yr', 
    #                          'emb91', 'Cardiac_case_pre_12', 'lab_max_hba1c_1yr', 'er_clm_cnt_2yr', 'emb2', 'emb144', 'emb98', 'racial_diversity', 'emb16', 
    #                          'emb159', 'Endocrine_Metabolic_case_pre_12', 'emb85', 'clm_ln_cnt_1yr', 'emb123', 'hpd_count', 'emb68', 'emb49', 'pharmacy_gpi4_1yr_count', 
    #                          'card_visits_pre_12'],
    #     machine_type={"machine_type": "n2-standard-64"},
    #     starting_replica_count=1,
    #     max_replica_count=10,
    # )

    # digestive_inference = vertex_batch_predict_bigquery_component(
    #     project=project_id,
    #     service_account=service_account,
    #     cmek_key=cmek_key,
    #     location=region,
    #     cost_center= constants["COSTCENTER"],
    #     owner=constants["OWNER"],
    #     model_resource_name="projects/46378383599/locations/us-east4/models/6058832436580581376",
    #     key_field="individual_id", 
    #     input_table="anbc-hcb-dev.clin_analytics_hcb_dev.cp_strategic_stratification_test_analytic",  # project.dataset.table
    #     output_table="anbc-hcb-dev.clin_analytics_hcb_dev.cp_strategic_stratification_test_digestive_model", # project.dataset.table (final output)
    #     compute_dataset="hcm_cm_de_beam_dev_hcm_cm_de", #hcm_cm_de_dec_beam_{ENV}_hcm_cm_de"
    #     ## add function for how to reorder table
    #     expiration_days=30, # output table expiration in days
    #     excluded_fields=["individual_id"],  # Fields to include
    #     selected_features = ['pharmacy_days_supply_1yr_cnt', 'colorectal_cancer', 'emb66', 'inflammatory_bowel_disease', 'emb211', 'ortho_visits_pre_12', 'emb110', 
    #                          'cost_0_3', 'emb39', 'uniq_rev_cd_cnt_1yr', 'emb154', 'emb205', 'air_qualtiy', 'emb158', 'emb118', 'change_q1_minus_q_2', 
    #                          'Renal_case_pre_12', 'emb233', 'emb89', 'emb64', 'emb2', 'emb62', 'emb246', 'emb65', 'emb169', 'emb199', 'emb208', 'lab_visit_cnt_6mo', 
    #                          'emb226', 'er_clm_cnt_2yr', 'uniq_rev_cd_cnt_6mo', 'emb1', 'emb71', 'Cardiac_case_pre_12', 'emb70', 'gastro_visits_pre_3', 'emb77', 
    #                          'gastro_visits_pre_9', 'emb84', 'dx_cd_cnt_6mo', 'emb196', 'emb37', 'pharmacy_disp_6mo_cnt', 'emb17', 'iron_deficiency_anemia', 'emb112', 
    #                          'emb197', 'gastro_visits_pre_1', 'emb103', 'emb31', 'emb41', 'emb45', 'emb109', 'Digestive_case_pre_6', 'cancer', 'proactive_health', 
    #                          'emb43', 'emb32', 'Injury_Poisoning_case_pre_6', 'pharmacy_days_supply_2yr_cnt', 'emb11', 'emb122', 'clm_ln_cnt_3mo', 'emb253', 'emb104', 
    #                          'emb107', 'emb222', 'emb24', 'Digestive_case_pre_3', 'bh', 'clm_ln_cnt_1yr', 'emb212', 'emb149', 'uniq_dx_cd_cnt_3mo', 'region_missing', 
    #                          'emb251', 'emb49', 'emb44', 'Endocrine_Metabolic_case_pre_12', 'uniq_rev_cd_cnt_2yr', 'emb35', 'pcp_visits_pre_3', 'emb56', 'emb234', 
    #                          'emb164', 'Musculoskeletal_case_pre_12', 'uniq_dx_cd_cnt_1yr', 'gastro_visits_pre_6', 'emb125', 'Infectious_case_pre_12', 'region_regW', 
    #                          'emb136', 'lab_max_ggt_2yr', 'emb67', 'emb152', 'pcp_visits_pre_1', 'change_q1_minus_q_3', 'er_clm_cnt_6mo', 'housing_quality', 'emb85', 
    #                          'emb179', 'other_spec_visits_pre_3', 'Infectious_case_pre_3', 'gastro_cv', 'emb102', 'dx_diff', 'uniq_proc_cd_cnt_2yr', 'lab_max_bilirub_2yr', 
    #                          'obstructive_sleep_apnea', 'emb215', 'emb190', 'emb180', 'Respiratory_case_pre_12', 'emb4', 'emb225', 'Injury_Poisoning_case_pre_12',
    #                          'emb74', 'lab_visit_cnt_1yr', 'other_spec_visits_pre_1', 'emb12', 'emb82', 'emb188', 'pcp_visits_pre_6', 'cost_6_9', 'emb156', 'emb181', 
    #                          'Oncologic_case_pre_12', 'Obstetric_case_pre_12', 'emb27', 'age', 'emb61', 'pcp_visit_cnt_2yr', 'emb242', 'Neurologic_case_pre_12', 'emb191', 
    #                          'emb34', 'emb176', 'cholelithiasis_cholecystitis', 'emb250', 'cost_3_6', 'card_visits_pre_12', 'emb111', 'emb92', 'alcoholism', 'ortho_cv', 
    #                          'Digestive_case_pre_12', 'emb254', 'emb55', 'emb63', 'emb47', 'cataract', 'nonspecific_gastritis_dyspepsia', 'index_month', 'pancreatitis', 
    #                          'emb88', 'emb121', 'uniq_dx_cd_cnt_2yr', 'emb26', 'esophageal_cancer', 'emb182', 'gastro_visits_pre_12', 'diverticular_disease', 'emb171', 
    #                          'emb90', 'change_h_1_minus_h_2', 'hpd_count', 'emb79', 'emb170', 'emb95'],
    #     machine_type={"machine_type": "n2-standard-64"},
    #     starting_replica_count=1,
    #     max_replica_count=10,
    # )

    # endocrine_inference = vertex_batch_predict_bigquery_component(
    #     project=project_id,
    #     service_account=service_account,
    #     cmek_key=cmek_key,
    #     location=region,
    #     cost_center= constants["COSTCENTER"],
    #     owner=constants["OWNER"],
    #     model_resource_name="projects/46378383599/locations/us-east4/models/2324222445583597568",
    #     key_field="individual_id", 
    #     input_table="anbc-hcb-dev.clin_analytics_hcb_dev.cp_strategic_stratification_test_analytic",  # project.dataset.table
    #     output_table="anbc-hcb-dev.clin_analytics_hcb_dev.cp_strategic_stratification_test_endocrine_model", # project.dataset.table (final output)
    #     compute_dataset="hcm_cm_de_beam_dev_hcm_cm_de", #hcm_cm_de_dec_beam_{ENV}_hcm_cm_de"
    #     ## add function for how to reorder table
    #     expiration_days=30, # output table expiration in days
    #     excluded_fields=["individual_id"],  # Fields to include
    #     selected_features = ['endo_visits_pre_6', 'lab_max_psa_1yr', 'emb195', 'uniq_rev_cd_cnt_6mo', 'emb229', 'chronic_obstructive_pulmonary_disease', 
    #                          'ortho_visits_pre_6', 'lab_visit_cnt_6mo', 'emb234', 'emb230', 'emb233', 'Oncologic_case_pre_12', 'Obstetric_case_pre_12', 
    #                          'lab_visit_cnt_1yr', 'uniq_proc_cd_cnt_1yr', 'emb80', 'rx_diff', 'clm_ln_cnt_2yr', 'emb56', 'endo_visits_pre_9', 'emb190', 
    #                          'emb242', 'emb146', 'emb199', 'index_month', 'lab_max_hba1c_2yr', 'emb3', 'emb144', 'emb231', 'emb44', 'emb60', 'allergy', 
    #                          'language_score', 'emb128', 'emb110', 'uniq_rev_cd_cnt_1yr', 'emb246', 'lab_max_hba1c_1yr', 'emb244', 'pharmacy_gpi4_3mo_count', 
    #                          'emb51', 'Endocrine_Metabolic_case_pre_6', 'mem_mos_12_post', 'pharmacy_days_supply_6mo_cnt', 'lab_max_glucose_1yr', 'emb29', 
    #                          'pcp_visits_pre_9', 'endo_visits_pre_12', 'emb47', 'emb134', 'other_spec_visits_pre_6', 'card_visits_pre_12', 'emb183', 'emb67', 
    #                          'emb120', 'emb63', 'emb125', 'Neurologic_case_pre_12', 'mm_2yr_cnt', 'emb23', 'emb49', 'clm_ln_cnt_1yr', 'age', 'Renal_case_pre_12', 
    #                          'emb123', 'other_spec_visits_pre_1', 'emb94', 'emb232', 'Infectious_case_pre_12', 'emb96', 'emb107', 'obesity', 'other_spec_visits_pre_3', 
    #                          'emb7', 'emb168', 'emb193', 'dx_diff', 'emb197', 'lab_max_glucose_2yr', 'emb200', 'emb43', 'lab_visit_cnt_2yr', 'diverticular_disease', 
    #                          'emb17', 'Respiratory_case_pre_12', 'emb20', 'emb203', 'emb187', 'emb196', 'emb111', 'diabetes_mellitus', 'uniq_dx_cd_cnt_6mo', 'emb127', 
    #                          'emb75', 'Endocrine_Metabolic_case_pre_12', 'Injury_Poisoning_case_pre_12', 'cost_0_3', 'emb157', 'ortho_visits_pre_12', 
    #                          'Endocrine_Metabolic_case_pre_3', 'emb224', 'emb174', 'Cardiac_case_pre_12', 'endo_cv', 'cost_6_9', 'emb211', 'emb8', 
    #                          'uniq_proc_cd_cnt_6mo', 'emb34', 'emb2', 'emb182', 'emb194', 'emb202', 'crime_score', 'emb71', 'ortho_visits_pre_1', 'emb69', 
    #                          'Digestive_case_pre_12', 'emb31', 'emb243', 'emb176', 'emb98', 'cancer', 'chronic_thyroid_disorders', 'Musculoskeletal_case_pre_12'],
    #     machine_type={"machine_type": "n2-standard-64"},
    #     starting_replica_count=1,
    #     max_replica_count=10,
    # )

    # msk_inference = vertex_batch_predict_bigquery_component(
    #     project=project_id,
    #     service_account=service_account,
    #     cmek_key=cmek_key,
    #     location=region,
    #     cost_center= constants["COSTCENTER"],
    #     owner=constants["OWNER"],
    #     model_resource_name="projects/46378383599/locations/us-east4/models/5659137969651449856",
    #     key_field="individual_id", 
    #     input_table="anbc-hcb-dev.clin_analytics_hcb_dev.cp_strategic_stratification_test_analytic",  # project.dataset.table
    #     output_table="anbc-hcb-dev.clin_analytics_hcb_dev.cp_strategic_stratification_test_msk_model", # project.dataset.table (final output)
    #     compute_dataset="hcm_cm_de_beam_dev_hcm_cm_de", #hcm_cm_de_dec_beam_{ENV}_hcm_cm_de"
    #     ## add function for how to reorder table
    #     expiration_days=30, # output table expiration in days
    #     excluded_fields=["individual_id"],  # Fields to include
    #     selected_features = ['emb208', 'ortho_visits_pre_9', 'emb5', 'emb113', 'pharmacy_gpi4_3mo_count', 'ortho_cv', 'other_spec_visits_pre_6', 'emb250', 'emb187', 
    #                          'emb95', 'glaucoma', 'emb82', 'emb162', 'citizenship_index', 'emb203', 'health_infra', 'emb158', 'emb104', 'low_back_pain', 'emb83', 
    #                          'emb170', 'emb61', 'emb4', 'emb27', 'emb58', 'emb93', 'emb149', 'language_score', 'uniq_dx_cd_cnt_6mo', 'uniq_dx_cd_cnt_2yr', 
    #                          'Endocrine_Metabolic_case_pre_12', 'emb225', 'emb188', 'other_spec_visits_pre_3', 'cost_6_9', 'spec_visits_pre_9', 'bh', 'lab_visit_cnt_6mo', 
    #                          'clm_ln_cnt_3mo', 'uniq_rev_cd_cnt_1yr', 'emb55', 'emb207', 'pharmacy_gpi4_6mo_count', 'emb79', 'pharmacy_days_supply_6mo_cnt', 
    #                          'uniq_dx_cd_cnt_3mo', 'emb120', 'emb68', 'emb48', 'uniq_proc_cd_cnt_2yr', 'Neurologic_case_pre_12', 'emb249', 'emb156', 'emb179', 
    #                          'income_inequality', 'emb70', 'emb185', 'emb196', 'emb232', 'parkinsons_disease', 'change_h_1_minus_h_2', 'cancer_visits_pre_9', 
    #                          'emb78', 'pharmacy_gpi4_1yr_count', 'emb109', 'lab_visit_cnt_1yr', 'emb205', 'emb226', 'emb94', 'cost_3_6', 'Injury_Poisoning_case_pre_3', 
    #                          'pcp_cv', 'emb86', 'emb105', 'ortho_visits_pre_12', 'emb7', 'cancer', 'card_visits_pre_12', 'emb62', 'emb137', 'emb57', 'Renal_case_pre_12', 
    #                          'er_clm_cnt_1yr', 'emb246', 'emb195', 'pharmacy_days_supply_3mo_cnt', 'substances_related_disorders', 'emb122', 'emb107', 'emb242', 
    #                          'cost_0_3', 'Musculoskeletal_case_pre_3', 'pharmacy_gpi4_2yr_count', 'emb121', 'emb134', 'osteoporosis', 'ortho_visits_pre_1', 'depression', 
    #                          'emb72', 'emb194', 'air_qualtiy', 'emb59', 'emb42', 'Cardiac_case_pre_12', 'pcp_visit_cnt_1yr', 'emb33', 'emb157', 'emb77', 'emb118', 
    #                          'emb212', 'emb169', 'lab_max_hba1c_2yr', 'endo_cv', 'emb206', 'pulmo_visits_pre_9', 'Injury_Poisoning_case_pre_12', 
    #                          'Musculoskeletal_case_pre_12', 'emb214', 'Oncologic_case_pre_12', 'change_q1_minus_q_4', 'unemployment_index', 'emb45', 
    #                          'other_spec_visits_pre_9', 'emb235', 'dx_diff', 'emb22', 'gastro_visits_pre_6', 'Infectious_case_pre_12', 'emb240', 'change_q1_minus_q_3', 
    #                          'emb216', 'hpd_count', 'emb150', 'emb75', 'racial_diversity', 'uniq_dx_cd_cnt_1yr', 'emb154', 'emb238', 'osteoarthritis', 'spec_cv', 
    #                          'emb21', 'emb224', 'Respiratory_case_pre_12', 'mm_2yr_cnt', 'gender_cd', 'pharmacy_disp_2yr_cnt', 'clm_ln_cnt_1yr', 'emb130', 'emb119', 
    #                          'neuro_cv', 'dx_cd_cnt_6mo', 'emb91', 'emb85', 'emb253', 'emb220', 'emb176', 'other_spec_visits_pre_12', 'Obstetric_case_pre_12', 
    #                          'index_month', 'emb30', 'Digestive_case_pre_12', 'lab_visit_cnt_2yr', 'lab_visit_cnt_3mo', 'alcoholism', 'emb69', 'emb197', 'obesity', 
    #                          'emb56', 'ortho_visits_pre_3', 'age', 'emb227', 'emb132', 'emb204', 'pharmacy_days_supply_1yr_cnt', 'pharmacy_disp_1yr_cnt'],
    #     machine_type={"machine_type": "n2-standard-64"},
    #     starting_replica_count=1,
    #     max_replica_count=10,
    # )

    # neuro_inference = vertex_batch_predict_bigquery_component(
    #     project=project_id,
    #     service_account=service_account,
    #     cmek_key=cmek_key,
    #     location=region,
    #     cost_center= constants["COSTCENTER"],
    #     owner=constants["OWNER"],
    #     model_resource_name="projects/46378383599/locations/us-east4/models/1122887244982517760",
    #     key_field="individual_id", 
    #     input_table="anbc-hcb-dev.clin_analytics_hcb_dev.cp_strategic_stratification_test_analytic",  # project.dataset.table
    #     output_table="anbc-hcb-dev.clin_analytics_hcb_dev.cp_strategic_stratification_test_neuro_model", # project.dataset.table (final output)
    #     compute_dataset="hcm_cm_de_beam_dev_hcm_cm_de", #hcm_cm_de_dec_beam_{ENV}_hcm_cm_de"
    #     ## add function for how to reorder table
    #     expiration_days=30, # output table expiration in days
    #     excluded_fields=["individual_id"],  # Fields to include
    #     selected_features = ['pharmacy_gpi4_3mo_count', 'clm_ln_cnt_1yr', 'pharmacy_disp_1yr_cnt', 'lab_visit_cnt_2yr', 'emb62', 'emb134', 'emb247', 
    #                          'Musculoskeletal_case_pre_12', 'emb104', 'emb174', 'Respiratory_case_pre_12', 'emb154', 'hpd_major_flag', 'Endocrine_Metabolic_case_pre_12', 
    #                          'clm_ln_cnt_3mo', 'pharmacy_disp_6mo_cnt', 'urg_cv', 'emb31', 'emb188', 'emb2', 'emb113', 'neuro_cv', 'emb181', 'neuro_visits_pre_6', 
    #                          'Renal_case_pre_12', 'emb47', 'neuro_visits_pre_12', 'emb242', 'cost_6_9', 'emb218', 'emb74', 'endo_cv', 'dx_diff', 'emb200', 'emb129', 
    #                          'card_visits_pre_12', 'emb163', 'pharmacy_days_supply_3mo_cnt', 'emb225', 'cost_9_12', 'emb50', 'hpd_count', 'age', 'emb167', 
    #                          'other_spec_visits_pre_9', 'emb126', 'emb94', 'uniq_rev_cd_cnt_6mo', 'Cardiac_case_pre_12', 'index_month', 'uniq_dx_cd_cnt_6mo', 
    #                          'emb162', 'emb131', 'emb221', 'Obstetric_case_pre_12', 'Neurologic_case_pre_12', 'er_clm_cnt_2yr', 'emb101', 'change_h_1_minus_h_2', 
    #                          'emb224', 'Infectious_case_pre_12', 'emb33', 'cost_3_6', 'emb26', 'natural_disaster', 'Injury_Poisoning_case_pre_12', 'health_acess', 
    #                          'emb217', 'emb197', 'uniq_rev_cd_cnt_3mo', 'uniq_dx_cd_cnt_2yr', 'emb25', 'pharmacy_days_supply_2yr_cnt', 'emb81', 'cost_0_3', 
    #                          'lab_max_triglyc_1yr', 'epilepsy', 'emb65', 'emb246', 'emb34', 'emb118', 'Oncologic_case_pre_12', 'lab_max_bilirub_1yr', 'mm_1yr_cnt', 
    #                          'Neurologic_case_pre_3', 'emb106', 'Neurologic_case_pre_6', 'emb236', 'emb178', 'uniq_dx_cd_cnt_3mo', 'neuro_visits_pre_3', 'emb43', 'emb97', 
    #                          'Digestive_case_pre_12', 'uniq_proc_cd_cnt_6mo', 'mem_mos_12_post', 'emb139', 'emb173', 'emb45', 'multiple_sclerosis', 'depression', 'emb170', 
    #                          'emb0', 'neuro_visits_pre_9', 'parkinsons_disease', 'emb18', 'emb49', 'emb88', 'emb244', 'cerebrovascular_disease', 'dementia', 'emb109'],
    #     machine_type={"machine_type": "n2-standard-64"},
    #     starting_replica_count=1,
    #     max_replica_count=10,
    # )

    # resp_inference = vertex_batch_predict_bigquery_component(
    #     project=project_id,
    #     service_account=service_account,
    #     cmek_key=cmek_key,
    #     location=region,
    #     cost_center= constants["COSTCENTER"],
    #     owner=constants["OWNER"],
    #     model_resource_name="projects/46378383599/locations/us-east4/models/9052600288875118592",
    #     key_field="individual_id", 
    #     input_table="anbc-hcb-dev.clin_analytics_hcb_dev.cp_strategic_stratification_test_analytic",  # project.dataset.table
    #     output_table="anbc-hcb-dev.clin_analytics_hcb_dev.cp_strategic_stratification_test_resp_model", # project.dataset.table (final output)
    #     compute_dataset="hcm_cm_de_beam_dev_hcm_cm_de", #hcm_cm_de_dec_beam_{ENV}_hcm_cm_de"
    #     ## add function for how to reorder table
    #     expiration_days=30, # output table expiration in days
    #     excluded_fields=["individual_id"],  # Fields to include
    #     selected_features = ['Infectious_case_pre_3', 'emb153', 'Endocrine_Metabolic_case_pre_12', 'head_neck_cancer', 'emb139', 'emb92', 'emb100', 
    #                          'Infectious_case_pre_12', 'emb62', 'clm_ln_cnt_2yr', 'emb3', 'emb142', 'mem_mos_12_pre', 'emb89', 'emb227', 'emb46', 'pulmo_visits_pre_12', 
    #                          'emb219', 'emb108', 'emb129', 'emb31', 'index_month', 'Respiratory_case_pre_12', 'emb220', 'emb172', 'Neurologic_case_pre_12', 
    #                          'congential_heart_disease', 'pharmacy_gpi4_6mo_count', 'emb54', 'endo_cv', 'emb255', 'lab_max_altsgpt_2yr', 'emb52', 'emb35', 
    #                          'Musculoskeletal_case_pre_12', 'language_score', 'emb25', 'esophageal_cancer', 'emb199', 'emb94', 'uniq_dx_cd_cnt_2yr', 'emb96', 'emb59', 
    #                          'emb204', 'emb229', 'Oncologic_case_pre_3', 'mm_2yr_cnt', 'proactive_health', 'emb48', 'Digestive_case_pre_12', 'Renal_case_pre_12', 'emb69', 
    #                          'emb110', 'emb10', 'spec_visit_cnt_6mo', 'emb147', 'emb91', 'hpd_count', 'pharmacy_gpi4_3mo_count', 'oral_cancer', 'pulmo_visits_pre_9', 
    #                          'emb44', 'emb224', 'cancer_visits_pre_6', 'er_clm_cnt_3mo', 'uniq_rev_cd_cnt_1yr', 'emb23', 'emb6', 'food_access', 'emb7', 'emb101', 
    #                          'lab_visit_cnt_2yr', 'pharmacy_days_supply_6mo_cnt', 'change_q1_minus_q_4', 'emb70', 'card_visits_pre_12', 'uniq_dx_cd_cnt_3mo', 'emb192', 
    #                          'asthma', 'technology_access', 'uniq_dx_cd_cnt_1yr', 'emb47', 'emb182', 'Obstetric_case_pre_12', 'cost_6_9', 'uniq_rev_cd_cnt_3mo', 'emb16', 
    #                          'emb126', 'emb149', 'emb36', 'emb177', 'unemployment_index', 'emb240', 'Cardiac_case_pre_12', 'clm_ln_cnt_1yr', 'chronic_renal_failure', 
    #                          'pulmo_visits_pre_1', 'emb230', 'cost_3_6', 'age', 'emb88', 'dx_cd_cnt_6mo', 'emb235', 'emb215', 'emb63', 'emb98', 'emb17', 'emb196', 
    #                          'rx_diff', 'pharmacy_days_supply_3mo_cnt', 'emb211', 'emb166', 'emb60', 'emb55', 'Oncologic_case_pre_12', 'emb186', 'emb244', 'emb50', 
    #                          'emb53', 'emb170', 'pharmacy_gpi4_2yr_count', 'emb117', 'chronic_obstructive_pulmonary_disease', 'emb64', 'emb72', 'mem_mos_12_post', 
    #                          'emb49', 'citizenship_index', 'emb181', 'emb87', 'emb214', 'change_q1_minus_q_2', 'emb161', 'emb122', 'emb203', 'other_spec_cv', 'emb191', 
    #                          'cost_0_3', 'emb13', 'Injury_Poisoning_case_pre_12', 'lab_visit_cnt_6mo', 'mm_1yr_cnt', 'emb104', 'emb156', 'lab_max_hba1c_2yr', 'emb210', 
    #                          'emb136', 'emb127', 'spec_visit_cnt_3mo', 'emb21', 'emb33', 'clm_ln_cnt_3mo', 'emb248', 'emb150', 'emb141'],
    #     machine_type={"machine_type": "n2-standard-64"},
    #     starting_replica_count=1,
    #     max_replica_count=10,
    # )

    # run_claims.after(run_create_cohort)
    # parallel_tasks = [run_medical_case
    #                     ,run_HPD
    #                     ,run_physician_visits
    #                     ,run_cost_cv]
    # for t in parallel_tasks:
    #     t.after(run_claims)
    # run_analytic.after(*parallel_tasks)
    # cancer_inference.after(run_analytic)
    # cardiac_inference.after(cancer_inference)
    # digestive_inference.after(cardiac_inference)
    # endocrine_inference.after(digestive_inference)
    # msk_inference.after(endocrine_inference)
    # neuro_inference.after(msk_inference)
    # resp_inference.after(neuro_inference)
    #ptasks2 = [cancer_inference
    #            , cardiac_inference
    #            , digestive_inference
    #            , endocrine_inference
    #            , msk_inference
    #            , neuro_inference
    #            , resp_inference]
    #ptasks2.after(run_analytic)
    #run_combine.after(resp_inference)
    

In [17]:
compiler.Compiler().compile(
    pipeline_func=bigquery_pipeline,
    package_path="cp_strategic_strat_inference.json"
)

In [18]:
# Initialize Vertex AI client
aiplatform.init(
    project="anbc-dev-hcm-cm-de",  # Replace with your GCP project ID
    location="us-east4",           # Replace with your region
    service_account=service_account
)

In [19]:
# Define the pipeline job
pipeline_job = aiplatform.PipelineJob(
    display_name="strategic_strat_cp_bq",
    template_path="cp_strategic_strat_inference.json",
    pipeline_root="gs://hcm-cm-de-code-hcb-dev/mlops_cp_strategic_strat/",
    enable_caching=False,  # Replace with your GCS bucket path
    encryption_spec_key_name="projects/cvs-key-vault-nonprod/locations/us-east4/keyRings/gkr-nonprod-us-east4/cryptoKeys/gk-anbc-dev-hcm-cm-de-us-east4",  # Specify the CMEK
    parameter_values={
        "project_id": "anbc-dev-hcm-cm-de",
        "region": "us-east4",
        "cmek_key": "projects/cvs-key-vault-nonprod/locations/us-east4/keyRings/gkr-nonprod-us-east4/cryptoKeys/gk-anbc-dev-hcm-cm-de-us-east4"
    }
)

# Submit the pipeline job
pipeline_job.run(
    sync=True
)

Creating PipelineJob
PipelineJob created. Resource name: projects/46378383599/locations/us-east4/pipelineJobs/strategic-stratification-cp-bigquery-inference-pipeline-20260224102723
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/46378383599/locations/us-east4/pipelineJobs/strategic-stratification-cp-bigquery-inference-pipeline-20260224102723')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-east4/pipelines/runs/strategic-stratification-cp-bigquery-inference-pipeline-20260224102723?project=46378383599
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/strategic-stratification-cp-bigquery-inference-pipeline-20260224102723 current state:
3
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/strategic-stratification-cp-bigquery-inference-pipeline-20260224102723 current state:
3
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/strategic-stratification-cp-bigquery-inference-p